# 07 — TF-IDF Headline Source Classification (FOX vs NBC)

Concise experimental flow:
1) build dataset,
2) baseline linear models,
3) non-linear/ensemble alternatives,
4) cleaning ablation.

In [127]:
from pathlib import Path
import html, re, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score, train_test_split, GridSearchCV
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

SEED = int(np.random.default_rng().integers(0, 1_000_000_000))


In [128]:
ROOT = Path.cwd().resolve().parent
fox = pd.read_csv(ROOT / 'data/raw/fox_scraped_all.csv')
nbc = pd.read_csv(ROOT / 'data/raw/nbc_scraped_all.csv')

def clean_strict(x):
    if pd.isna(x): return ''
    s = unicodedata.normalize('NFKC', html.unescape(str(x))).lower()
    s = re.sub(r'https?://\S+|www\.\S+', ' ', s)
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def clean_light(x):
    if pd.isna(x): return ''
    s = unicodedata.normalize('NFKC', html.unescape(str(x))).lower()
    s = re.sub(r'https?://\S+|www\.\S+', ' ', s)
    s = re.sub(r"[^a-z0-9\s\-\'\!\?]", ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def clean_lower_ws(x):
    if pd.isna(x): return ''
    s = unicodedata.normalize('NFKC', html.unescape(str(x))).lower()
    return re.sub(r'\s+', ' ', s).strip()

def build_dataset(use_subtitle=False, cleaner=clean_strict, drop_cross_source_dupes=False):
    fox_text = fox['title'].fillna('').astype(str)
    nbc_text = nbc['title'].fillna('').astype(str)
    if use_subtitle:
        fox_text = (fox_text + ' ' + fox.get('subtitle', pd.Series(index=fox.index, dtype=str)).fillna('').astype(str)).str.strip()
        nbc_text = (nbc_text + ' ' + nbc.get('subtitle', pd.Series(index=nbc.index, dtype=str)).fillna('').astype(str)).str.strip()

    d = pd.concat([
        pd.DataFrame({'text': fox_text, 'source': 'FOX'}),
        pd.DataFrame({'text': nbc_text, 'source': 'NBC'})
    ], ignore_index=True)
    d['text_clean'] = d['text'].map(cleaner)
    d = d[d['text_clean'] != ''].drop_duplicates(['text_clean', 'source']).copy()

    if drop_cross_source_dupes:
        shared = d.groupby('text_clean')['source'].nunique()
        d = d[~d['text_clean'].isin(shared[shared > 1].index)]

    return d.reset_index(drop=True)

data = build_dataset(use_subtitle=False, cleaner=clean_strict)
X, y = data['text_clean'], data['source']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
data[['source']].value_counts().rename('count').to_frame().head()


,count
source,
FOX,2000
NBC,1799


## Baseline: TF-IDF + linear models

In [129]:
def fit_eval(pipe):
    pipe.fit(X_train, y_train)
    p = pipe.predict(X_test)
    return float(accuracy_score(y_test, p))

base_models = {
    'LogReg uni+bi': Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.95, sublinear_tf=True)), ('clf', LogisticRegression(max_iter=2000, random_state=SEED))]),
    'LinearSVC uni+bi': Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.95, sublinear_tf=True)), ('clf', LinearSVC(random_state=SEED))]),
    'SGD hinge uni+bi': Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.95, sublinear_tf=True)), ('clf', SGDClassifier(loss='hinge', max_iter=2000, tol=1e-3, random_state=SEED))]),
}

baseline_df = pd.DataFrame([{
    'experiment': k, 'accuracy': fit_eval(v)
} for k, v in base_models.items()]).sort_values('accuracy', ascending=False)
baseline_df


,experiment,accuracy
1,LinearSVC uni+bi,0.813158
0,LogReg uni+bi,0.805263
2,SGD hinge uni+bi,0.789474


## Alternative models (NB + ensembles)

In [130]:
alt_models = {
    'ComplementNB uni+bi': Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.95, sublinear_tf=True)), ('clf', ComplementNB(alpha=0.5))]),
    'MultinomialNB uni+bi': Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.95, sublinear_tf=True)), ('clf', MultinomialNB(alpha=0.5))]),
    'RandomForest + SVD': Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.95, sublinear_tf=True)), ('svd', TruncatedSVD(n_components=300, random_state=SEED)), ('clf', RandomForestClassifier(n_estimators=400, min_samples_leaf=2, random_state=SEED, n_jobs=-1))]),
    'ExtraTrees + SVD': Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.95, sublinear_tf=True)), ('svd', TruncatedSVD(n_components=300, random_state=SEED)), ('clf', ExtraTreesClassifier(n_estimators=500, min_samples_leaf=2, random_state=SEED, n_jobs=-1))]),
}

alt_df = pd.DataFrame([{
    'experiment': k, 'accuracy': fit_eval(v)
} for k, v in alt_models.items()]).sort_values('accuracy', ascending=False)
alt_df


,experiment,accuracy
0,ComplementNB uni+bi,0.806579
1,MultinomialNB uni+bi,0.805263
2,RandomForest + SVD,0.756579
3,ExtraTrees + SVD,0.747368


## Cleaning ablation (fixed model)

In [131]:
def score_cleaning(name, cleaner, use_subtitle=True, drop_cross_source_dupes=False, balanced=False):
    d = build_dataset(use_subtitle=use_subtitle, cleaner=cleaner, drop_cross_source_dupes=drop_cross_source_dupes)
    Xc, yc = d['text_clean'], d['source']
    clf = LinearSVC(random_state=SEED, class_weight='balanced' if balanced else None)
    pipe = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.95, sublinear_tf=True)), ('clf', clf)])

    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
    cv_scores = cross_val_score(pipe, Xc, yc, cv=cv, scoring='accuracy', n_jobs=-1)

    Xt, Xh, yt, yh = train_test_split(Xc, yc, test_size=0.2, stratify=yc, random_state=SEED)
    pipe.fit(Xt, yt)
    holdout = float(accuracy_score(yh, pipe.predict(Xh)))

    return {'setup': name, 'rows': len(d), 'cv_mean': float(np.mean(cv_scores)), 'cv_std': float(np.std(cv_scores)), 'holdout_acc': holdout}

cleaning_setups = [
    ('strict', clean_strict, True, False, False),
    ('light', clean_light, True, False, False),
    ('lower+ws only', clean_lower_ws, True, False, False),
    ('light + drop cross-source dupes', clean_light, True, True, False),
    ('light + drop dupes + balanced', clean_light, True, True, True),
]

cleaning_df = pd.DataFrame([
    score_cleaning(name, cleaner, subtitle, drop_dupes, bal)
    for name, cleaner, subtitle, drop_dupes, bal in cleaning_setups
]).sort_values('cv_mean', ascending=False)
cleaning_df


,setup,rows,cv_mean,cv_std,holdout_acc
2,lower+ws only,3799,0.884007,0.010672,0.867105
0,strict,3799,0.883832,0.011088,0.869737
1,light,3799,0.883832,0.011088,0.869737
3,light + drop cross-source dupes,3799,0.883832,0.011088,0.869737
4,light + drop dupes + balanced,3799,0.883481,0.012478,0.868421


In [132]:
summary = pd.DataFrame([
    {'block': 'baseline best', 'accuracy': float(baseline_df['accuracy'].max())},
    {'block': 'alternative best', 'accuracy': float(alt_df['accuracy'].max())},
    {'block': 'cleaning best holdout', 'accuracy': float(cleaning_df['holdout_acc'].max())},
])
summary


,block,accuracy
0,baseline best,0.813158
1,alternative best,0.806579
2,cleaning best holdout,0.869737


## Part 10 — Compact grid search (TF-IDF + linear model)

Small, focused search over strong TF-IDF + linear settings. We optimize CV accuracy and then evaluate the best model on the same holdout split.

In [133]:
grid_pipe = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LinearSVC(random_state=SEED)),
])

param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2), (1, 3)],
    "tfidf__min_df": [1, 2, 3],
    "tfidf__max_df": [0.9, 0.95, 1.0],
    "tfidf__sublinear_tf": [True, False],
    "clf__C": [0.25, 0.5, 1.0, 2.0],
}

grid_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEED)
grid = GridSearchCV(
    estimator=grid_pipe,
    param_grid=param_grid,
    cv=grid_cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=0,
    return_train_score=False,
)
grid.fit(X_train, y_train)

top10 = (
    pd.DataFrame(grid.cv_results_)
    .sort_values("mean_test_score", ascending=False)
    .loc[:, ["mean_test_score", "std_test_score", "param_tfidf__ngram_range", "param_tfidf__min_df", "param_tfidf__max_df", "param_tfidf__sublinear_tf", "param_clf__C"]]
    .head(10)
    .reset_index(drop=True)
)
top10

,mean_test_score,std_test_score,param_tfidf__ngram_range,param_tfidf__min_df,param_tfidf__max_df,param_tfidf__sublinear_tf,param_clf__C
0,0.806352,0.012177,"(1, 2)",1,0.90,True,1.0
1,0.806352,0.012177,"(1, 2)",1,1.00,True,1.0
2,0.806352,0.012177,"(1, 2)",1,0.95,True,1.0
3,0.805859,0.012886,"(1, 2)",1,0.95,False,1.0
4,0.805859,0.012886,"(1, 2)",1,1.00,False,1.0
5,0.805859,0.012886,"(1, 2)",1,0.90,False,1.0
6,0.805365,0.014800,"(1, 2)",1,0.90,True,0.5
7,0.805365,0.014800,"(1, 2)",1,1.00,True,0.5
8,0.805365,0.014800,"(1, 2)",1,0.95,True,0.5
9,0.805036,0.014036,"(1, 3)",1,0.95,True,2.0


In [134]:
best_grid_model = grid.best_estimator_
grid_holdout_acc = float(accuracy_score(y_test, best_grid_model.predict(X_test)))

pd.DataFrame([
    {"metric": "grid best CV mean", "value": float(grid.best_score_)},
    {"metric": "grid holdout acc", "value": grid_holdout_acc},
    {"metric": "current baseline best holdout", "value": float(baseline_df["accuracy"].max())},
])

,metric,value
0,grid best CV mean,0.806352
1,grid holdout acc,0.826316
2,current baseline best holdout,0.813158


## Part 11 — Joint tuning: cleaning strategy + TF-IDF hyperparameters

This combines cleaning choice with model tuning. For each cleaning strategy, we run the same compact `GridSearchCV` on `LinearSVC + TF-IDF`, then compare best CV and holdout.

In [135]:
joint_cleaners = {
    "strict": clean_strict,
    "light": clean_light,
    "lower+ws only": clean_lower_ws,
}

joint_param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2), (1, 3)],
    "tfidf__min_df": [1, 2, 3],
    "tfidf__max_df": [0.9, 0.95, 1.0],
    "tfidf__sublinear_tf": [True, False],
    "clf__C": [0.25, 0.5, 1.0, 2.0],
}

joint_rows = []

for cname, cleaner in joint_cleaners.items():
    d = build_dataset(use_subtitle=True, cleaner=cleaner, drop_cross_source_dupes=False)
    Xj, yj = d["text_clean"], d["source"]

    Xj_train, Xj_test, yj_train, yj_test = train_test_split(
        Xj, yj, test_size=0.2, stratify=yj, random_state=SEED
    )

    pipe = Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("clf", LinearSVC(random_state=SEED)),
    ])

    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEED)
    gs = GridSearchCV(
        estimator=pipe,
        param_grid=joint_param_grid,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
        verbose=0,
    )
    gs.fit(Xj_train, yj_train)

    holdout = float(accuracy_score(yj_test, gs.best_estimator_.predict(Xj_test)))
    joint_rows.append(
        {
            "cleaning": cname,
            "rows": len(d),
            "best_cv": float(gs.best_score_),
            "holdout_acc": holdout,
            "best_params": gs.best_params_,
        }
    )

joint_df = pd.DataFrame(joint_rows).sort_values("holdout_acc", ascending=False)
joint_df

,cleaning,rows,best_cv,holdout_acc,best_params
0,strict,3799,0.879565,0.871053,"{'clf__C': 2.0, 'tfidf__max_df': 0.9, 'tfidf__..."
1,light,3799,0.879565,0.871053,"{'clf__C': 2.0, 'tfidf__max_df': 0.9, 'tfidf__..."
2,lower+ws only,3799,0.880223,0.871053,"{'clf__C': 2.0, 'tfidf__max_df': 0.9, 'tfidf__..."


In [136]:
if "joint_df" not in globals() or joint_df.empty:
    raise RuntimeError("Run the previous Part 11 setup cell first (it creates `joint_df`).")

best_joint = joint_df.iloc[0]

pd.DataFrame([
    {"metric": "best joint cleaning", "value": best_joint["cleaning"]},
    {"metric": "best joint CV", "value": float(best_joint["best_cv"])},
    {"metric": "best joint holdout", "value": float(best_joint["holdout_acc"])},
    {"metric": "part 10 holdout", "value": float(grid_holdout_acc)},
    {"metric": "baseline best holdout", "value": float(baseline_df["accuracy"].max())},
])

,metric,value
0,best joint cleaning,strict
1,best joint CV,0.879565
2,best joint holdout,0.871053
3,part 10 holdout,0.826316
4,baseline best holdout,0.813158
